In [24]:
import os.path
import matlab.engine
from custom_io import open_dir, open_file
import numpy as np


In [4]:
def load_nd2_timestamps(
        nikon_ts_path: str, matlab_2p_folder: str, nargout: int = 3
) -> dict:
    """Run the belt processing pipeline in matlab-2p with extra output parameters,
    with the belt file, nikon metadata file, and matlab-2p scripts folder.
    The structure of a labview output file, decoded from the labview file
    (Movementdetection.vi, Integrator.vi) by column:
    1.  Rounds
    2.  Speed
    3.  Total distance
    4.  DistancePR (per round)
    5.  Reflectivity
    6.  Lick detection
    7.  Stripes in total
    8.  Stripes in round
    9.  Total time
    10. Time per round
    11. Stimuli
    ... Stimuli
    19. Stimuli
    20. Pupil area
    Args:
        belt_path (str): _description_
        nikon_ts_path (str): _description_
        matlab_2p_folder (str): _description_
        nargout (int, optional): _description_. Defaults to 3.

    Raises:
        Exception: _description_

    Returns:
        dict: _description_
    """
    eng = matlab.engine.start_matlab()
    # dialog window pops up in background!
    if matlab_2p_folder is None:
        matlab_2p_folder = open_dir("Open matlab-2p folder")
    m2p_path = eng.genpath(matlab_2p_folder)
    eng.addpath(m2p_path, nargout=0)
    nikon_fname = os.path.splitext(os.path.split(nikon_ts_path)[-1])[0]
    nikon_dir = os.path.split(nikon_ts_path)[0]
    return eng.openNikonTimeStamps(
        nikon_dir, nikon_fname, nargout=nargout
    )

In [36]:
res, _, _ = load_nd2_timestamps(nikon_ts_path="C:/sciebo/Test/example_with_lfp/T370_ChR2_d29_elec_002_nik.txt", matlab_2p_folder="D:/PhD/SDAnalysis/sdanalysis/m2p/", nargout = 3)

In [25]:
def _drop_useless_dimensions(array):
    """
    This function solves the problem that seems to stem from different Matlab versions used in belt processing.
    Depending on matlab version, the returned 1d array might turn into 2d: the shape of the array (x,) becomes (1,x). In terms of array elements,
    [x0, x1, ...] becomes [[x0, x1, ...]].
    This function detects if such a bad formatting occurred and attempts to correct it
    :param array: input array of matlab origin
    :return: the same data but with redundant dimension removed.
    """
    if len(array.shape) == 1:
        return array
    elif len(array.shape) == 2 and array.shape[0] == 1:
        # print(f"Matlab possibly messed up an array, shape {array.shape} detected; should probably be ({array.shape[1]},). attempting to remove the redundant dimension...")
        return array[0]

In [26]:
def _matlab_array_to_numpy_array( matlab_array):
        if type(matlab_array) is np.ndarray:
            return _drop_useless_dimensions(matlab_array)
        else:
            return _drop_useless_dimensions(np.array(matlab_array._data))

In [50]:
res.size

(1, 1)

In [51]:
import pandas as pd

In [73]:
df = pd.read_excel("C:\\sciebo\\Test\\example_with_lfp\\T370_ChR2_d29_elec_002_nik_read_out.xlsx", header=None)
df.rename(columns={0: "Time [m:s.ms]", 1: "SW Time [s]", 2: "NIDAQ Time [s]", 3: "Index"}, inplace=True)

In [75]:
df.to_excel("C:\\sciebo\\Test\\example_with_lfp\\T370_ChR2_d29_elec_002_nik_recorded_frames.xlsx", index=False)

In [59]:
df_readout = pd.read_table("C:\\sciebo\\Test\\example_with_lfp\\T370_ChR2_d29_elec_002_nik.txt", header=0, encoding="utf-16", sep="\t")